In [1]:
import csv
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
base = "CME"
threshold = 15

CME_track = "Input Data/ap2_CME.csv"
Dino_track = "Input Data/ap2_Dino.csv"
CME_detections = "Input Data/detections_CME.csv"
Dino_detections = "Input Data/detections_Dino.csv"

In [3]:
df_CME_track = pd.read_csv(CME_track, header = None)
M_CME_track = df_CME_track.to_numpy()
M_CME_track[:, 2] += M_CME_track[:, 1]-1
M_CME_track[:, 5] = 169 - M_CME_track[:, 5]

df_Dino_track = pd.read_csv(Dino_track)
df_Dino_track = df_Dino_track.drop("max_intensity", axis = 1)
M_Dino_track = df_Dino_track.to_numpy()
M_Dino_track[:, 1:3] += 1

df_CME_detections = pd.read_csv(CME_detections, header = None)
M_CME_detections = df_CME_detections.to_numpy()
#M_CME[:, 4] = M_CME[:, 4]-1

df_Dino_detections = pd.read_csv(Dino_detections)
df_Dino_detections = df_Dino_detections[["timepoint","label_id","x","y","z","intensity","max_intensity"]]
df_Dino_detections = df_Dino_detections.drop("max_intensity", axis=1)
M_Dino_detections = df_Dino_detections.to_numpy()
M_Dino_detections[:, 0] += 1


if base == "CME":
    comp_list = np.zeros((len(M_CME_track),33), dtype=object)
    M_x = M_CME_track
    M_y = M_Dino_track
    sec = "Dino"
if base == "Dino":
    comp_list = np.zeros((len(M_Dino_track),33), dtype=object)
    M_x = M_Dino_track
    M_y = M_CME_track
    sec = "CME"

M_z = M_CME_detections
M_w = M_Dino_detections

In [4]:
t_y = defaultdict(list)
t_z = defaultdict(list)
t_w = defaultdict(list)

for vec_y in M_y:
    t_y[vec_y[2]].append(vec_y)
for t_val in t_y:
    t_y[t_val] = np.array(t_y[t_val])

for vec_z in M_z:
    t_z[vec_z[0]].append(vec_z)
for t_val in t_z:
    t_z[t_val] = np.array(t_z[t_val])

for vec_w in M_w:
    t_w[vec_w[0]].append(vec_w)
for t_val in t_w:
    t_w[t_val] = np.array(t_w[t_val])

for i, vec_x in enumerate(M_x):
    t_val = vec_x[2]
    y_group = t_y[t_val]
    diffs = y_group[:, 3:6] - vec_x[3:6]
    dists = np.linalg.norm(diffs, axis=1)
    below = np.where(dists < threshold)[0]

    z_group = t_z[t_val]
    z_diffs = z_group[:, 2:5] - vec_x[3:6]
    z_dists = np.linalg.norm(z_diffs, axis=1)
    z_below = np.where(z_dists < threshold)[0]

    w_group = t_w[t_val]
    w_diffs = w_group[:, 2:5] - vec_x[3:6]
    w_dists = np.linalg.norm(w_diffs, axis=1)
    w_below = np.where(w_dists < threshold)[0]

    if len(below) > 0:
        y_ids = ";".join(str(int(y)) for y in y_group[below, 0])
        y_TL = ";".join(str(y) for y in y_group[below, 7])
        y_x = ";".join(str(y) for y in y_group[below, 3])
        y_y = ";".join(str(y) for y in y_group[below, 4])
        y_z = ";".join(str(y) for y in y_group[below, 5])
        y_FI = ";".join(str(y) for y in y_group[below, 6])
        dists_str = ";".join(f"{dists[j]:.2f}" for j in below)
    else:
        y_ids = ""
        y_TL = ""
        y_x = ""
        y_y = ""
        y_z = ""
        y_FI = ""
        dists_str = ""

    if len(z_below) > 0:
        z_ids = ";".join(str(z) for z in z_group[z_below, 1])
        z_x = ";".join(str(z) for z in z_group[z_below, 2])
        z_y = ";".join(str(z) for z in z_group[z_below, 3])
        z_z = ";".join(str(z) for z in z_group[z_below, 4])
        z_FI = ";".join(str(z) for z in z_group[z_below, 5])
        z_dists_str = ";".join(f"{z_dists[j]:.2f}" for j in z_below)
    else:
        z_ids = ""
        z_x = ""
        z_y = ""
        z_z = ""
        z_FI = ""
        z_dists_str = ""

    if len(w_below) > 0:
        w_ids = ";".join(str(w) for w in w_group[w_below, 1])
        w_x = ";".join(str(w) for w in w_group[w_below, 2])
        w_y = ";".join(str(w) for w in w_group[w_below, 3])
        w_z = ";".join(str(w) for w in w_group[w_below, 4])
        w_FI = ";".join(str(w) for w in w_group[w_below, 5])
        w_dists_str = ";".join(f"{w_dists[j]:.2f}" for j in w_below)
    else:
        w_ids = ""
        w_x = ""
        w_y = ""
        w_z = ""
        w_FI = ""
        w_dists_str = ""

    comp_list[i, 0:8] = vec_x
    comp_list[i, 8:27] = y_ids, y_TL, y_x, y_y, y_z, y_FI, dists_str, z_ids, z_x, z_y, z_z, z_FI, z_dists_str, w_ids, w_x, w_y, w_z, w_FI, w_dists_str

comp_list = np.delete(comp_list, 1, axis=1)
comp_list[:, [0, 1]] = comp_list[:, [1, 0]]
cols = list(range(comp_list.shape[1]))
col = cols.pop(6)
cols.insert(2, col)
comp_list = comp_list[:, cols]

In [5]:
start_col = 26
xmin_col, xmax_col, ymin_col, ymax_col, zmin_col, zmax_col = range(start_col, start_col + 6)

id_col = 1
cols1 = [2, 9, 15, 21]
cols2 = [3, 10, 16, 22]
cols3 = [4, 11, 17, 23]

ids = np.unique(comp_list[:, id_col])

for id_val in ids:
    group_idx = np.where(comp_list[:, id_col] == id_val)[0]
    group = comp_list[group_idx]

    vals1 = []
    for col in cols1:
        for cell in group[:, col]:
            vals1.extend([float(x) for x in str(cell).split(';') if x != ''])

    vals2 = []
    for col in cols2:
        for cell in group[:, col]:
            vals2.extend([float(x) for x in str(cell).split(';') if x != ''])

    vals3 = []
    for col in cols3:
        for cell in group[:, col]:
            vals3.extend([float(x) for x in str(cell).split(';') if x != ''])

    # Sätt extrema värden i de redan existerande kolumnerna
    comp_list[group_idx, xmin_col] = min(vals1) if vals1 else np.nan
    comp_list[group_idx, xmax_col] = max(vals1) if vals1 else np.nan
    comp_list[group_idx, ymin_col] = min(vals2) if vals2 else np.nan
    comp_list[group_idx, ymax_col] = max(vals2) if vals2 else np.nan
    comp_list[group_idx, zmin_col] = min(vals3) if vals3 else np.nan
    comp_list[group_idx, zmax_col] = max(vals3) if vals3 else np.nan

In [6]:
df_comp = pd.DataFrame(comp_list, columns= ["t", f"ID {base}", f"Track Length {base}", f"x {base}", f"y {base}", f"z {base}", f"FI {base}", f"ID {sec} Track", f"Track Length {sec} Track", f"x {sec} Track", f"y {sec} Track", f"z {sec} Track", f"FI {sec} Track", f"Distance {sec} Track", f"ID {base} Detections", f"x {base} Detections", f"y {base} Detections", f"z {base} Detections", f"FI {base} Detections", f"Distance {base} Detections", f"ID {sec} Detections", f"x {sec} Detections", f"y {sec} Detections", f"z {sec} Detections", f"FI {sec} Detections", f"Distance {sec} Detections", "xmin", "xmax", "ymin", "ymax", "zmin", "zmax"])

In [7]:
df_comp.to_csv(f"{base} Base Output/IDCoordinates.csv")

In [21]:
df_comp = df_comp[["t", f"ID {base}", f"Track Length {base}", f"x {base}", f"y {base}", f"z {base}", f"FI {base}", f"ID {sec} Track", f"Track Length {sec} Track", f"x {sec} Track", f"y {sec} Track", f"z {sec} Track", f"FI {sec} Track", f"Distance {sec} Track"]]

filtered = df_comp[df_comp["Track Length CME"]==99]



In [22]:
#filtered.to_csv("multicomp_TL99.csv", index=False)